# Analyse approfondie automatique des modèles candidats

Ce notebook automatise l'exploration détaillée de tous les modèles présents dans `full_results`.

In [15]:
from pathlib import Path
import re
import json
import pickle
import os
import sys
import hashlib
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src.model.kmeans import *
from src.model.cluster_analyzing import *
from src.model.model_evaluation import *
from src.features.features_groups import *


sns.set_theme(style="whitegrid")

OUTPUT_DIR = Path("../analysis_outputs/full_results_deep_dive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUTS_FILTERED_PATH = Path("../data/final/model_outputs/filter/comparison_table.csv")
FULL_RESULTS_PKL_PATH = Path("../data/final/model_outputs/filter/all_results.pkl")



DISPLAY_ALL_MODELS = True
MAX_MODELS_TO_DISPLAY = 23

print("Output dir:", OUTPUT_DIR.resolve())

Output dir: C:\Projets_ETS\MTI830\clustering_chess_player_lichess-1\analysis_outputs\full_results_deep_dive


In [16]:
if "full_results" in globals():
    print("Variable `full_results` trouvée en mémoire.")
else:
    
    if FULL_RESULTS_PKL_PATH.exists():
        with open(FULL_RESULTS_PKL_PATH, "rb") as f:
            full_results = pickle.load(f)
        print(f"`full_results` chargé depuis: {FULL_RESULTS_PKL_PATH}")
    else:
        raise FileNotFoundError(
            "Aucune variable `full_results` en mémoire et aucun fichier pickle trouvé."
        )

print("Nombre de modèles dans full_results :", len(full_results))
print("Clés disponibles :", list(full_results[0].keys()))

Variable `full_results` trouvée en mémoire.
Nombre de modèles dans full_results : 23
Clés disponibles : ['feature_set_name', 'k', 'seed', 'features', 'labels', 'model', 'df_with_clusters', 'cluster_sizes', 'clustering_summary_mean', 'clustering_summary_median', 'performance_summary_mean', 'performance_summary_median', 'progression_summary_mean', 'progression_summary_median', 'standardized_profiles', 'silhouette', 'inertia', 'imbalance_ratio', 'elo_slope_per_game_range', 'elo_slope_per_game_std_between_clusters', 'elo_slope_per_game_std_between_clusters_weighted', 'elo_slope_per_game_std_within_clusters_weighted']


In [17]:
def sanitize_name(text: str, max_len: int = 40) -> str:
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9_\-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text[:max_len]


def make_model_id(result: dict) -> str:
    # nom court + hash → évite chemins trop longs
    short_name = sanitize_name(result["feature_set_name"], max_len=30)
    unique_str = f"{result['feature_set_name']}__k{result['k']}__seed{result['seed']}"
    short_hash = hashlib.md5(unique_str.encode()).hexdigest()[:8]
    return f"{short_name}_k{result['k']}_s{result['seed']}_{short_hash}"


def save_dataframe(df, path: Path):
    path = path.resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(df, pd.Series):
        df.to_csv(path, header=True)
    else:
        df.to_csv(path, index=True)


def save_json(obj: dict, path: Path):
    path = path.resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def save_plot(fig, path: Path, display_outputs: bool = True):
    path = path.resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    if display_outputs:
        plt.show()
    plt.close(fig)
    return path


def build_model_summary_from_result(result: dict) -> dict:
    cluster_sizes = result["cluster_sizes"]
    return {
        "feature_set_name": result["feature_set_name"],
        "k": int(result["k"]),
        "best_seed": int(result["seed"]),
        "n_features": int(len(result["features"])),
        "features": result["features"],
        "silhouette": float(result.get("silhouette", np.nan)),
        "inertia": float(result.get("inertia", np.nan)),
        "min_cluster_size": int(cluster_sizes.min()),
        "max_cluster_size": int(cluster_sizes.max()),
        "imbalance_ratio": float(result.get("imbalance_ratio", np.nan)),
        "elo_slope_per_game_range": float(result.get("elo_slope_per_game_range", np.nan)),
        "elo_slope_per_game_std_between_clusters": float(result.get("elo_slope_per_game_std_between_clusters", np.nan)),
        "elo_slope_per_game_std_between_clusters_weighted": float(result.get("elo_slope_per_game_std_between_clusters_weighted", np.nan)),
        "elo_slope_per_game_std_within_clusters_weighted": float(result.get("elo_slope_per_game_std_within_clusters_weighted", np.nan)),
    }


In [25]:
def render_and_save_model_outputs(result: dict, output_root: Path, display_outputs: bool = True):
    output_root = output_root.resolve()

    model_id = make_model_id(result)
    model_dir = output_root / model_id
    tables_dir = model_dir / "tables"
    plots_dir = model_dir / "plots"
    data_dir = model_dir / "data"

    tables_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)

    feature_set_name = result["feature_set_name"]
    chosen_k = int(result["k"])
    final_dataset_clustered = result["df_with_clusters"]

    analysis = {
        "cluster_sizes": result["cluster_sizes"],
        "clustering_summary_mean": result["clustering_summary_mean"],
        "clustering_summary_median": result["clustering_summary_median"],
        "performance_summary_mean": result["performance_summary_mean"],
        "performance_summary_median": result["performance_summary_median"],
        "progression_summary_mean": result["progression_summary_mean"],
        "progression_summary_median": result["progression_summary_median"],
        "standardized_profiles": result["standardized_profiles"],
    }

    model_summary = build_model_summary_from_result(result)

    # ---------- SAVE TABLES ----------
    save_dataframe(analysis["cluster_sizes"], tables_dir / "cluster_sizes.csv")
    save_dataframe(analysis["clustering_summary_mean"], tables_dir / "clustering_summary_mean.csv")
    save_dataframe(analysis["clustering_summary_median"], tables_dir / "clustering_summary_median.csv")
    save_dataframe(analysis["performance_summary_mean"], tables_dir / "performance_summary_mean.csv")
    save_dataframe(analysis["performance_summary_median"], tables_dir / "performance_summary_median.csv")
    save_dataframe(analysis["progression_summary_mean"], tables_dir / "progression_summary_mean.csv")
    save_dataframe(analysis["progression_summary_median"], tables_dir / "progression_summary_median.csv")
    save_dataframe(analysis["standardized_profiles"], tables_dir / "standardized_profiles.csv")
    save_dataframe(final_dataset_clustered, data_dir / "final_dataset_clustered.csv")
    save_json(model_summary, model_dir / "model_summary.json")

    if display_outputs:
        display(Markdown(f"## {feature_set_name} — k={chosen_k} — seed={result['seed']}"))
        display(pd.DataFrame([model_summary]))
        display(Markdown("### Taille des clusters"))
        display(analysis["cluster_sizes"])
        display(Markdown("### Moyennes de progression"))
        display(analysis["progression_summary_mean"])
        display(Markdown("### Médianes de progression"))
        display(analysis["progression_summary_median"])
        display(Markdown("### Profils moyens"))
        display(analysis["clustering_summary_mean"])

    # ---------- SAVE PLOTS ----------
    fig = plt.figure(figsize=(12, 6))
    sns.heatmap(
        analysis["standardized_profiles"],
        annot=True,
        cmap="coolwarm",
        center=0,
    )
    plt.title(f"Standardized cluster profiles - {feature_set_name} (k={chosen_k})")
    plt.tight_layout()
    save_plot(fig, plots_dir / "standardized_profiles_heatmap.png", display_outputs=display_outputs)

    cluster_sizes = analysis["cluster_sizes"]
    fig = plt.figure(figsize=(6, 4))
    cluster_sizes.plot(kind="bar")
    plt.title(f"Cluster sizes - {feature_set_name} (k={chosen_k})")
    plt.ylabel("Number of players")
    plt.tight_layout()
    save_plot(fig, plots_dir / "cluster_sizes_barplot.png", display_outputs=display_outputs)

    progression_means = analysis["progression_summary_mean"]["elo_slope_per_game"]
    progression_stds = final_dataset_clustered.groupby("cluster")["elo_slope_per_game"].std()

    fig = plt.figure(figsize=(8, 5))
    plt.bar(
        progression_means.index.astype(str),
        progression_means.values,
        yerr=progression_stds.values,
        capsize=5,
    )
    plt.title(f"Mean elo_slope_per_game by cluster - {feature_set_name} (k={chosen_k})")
    plt.xlabel("Cluster")
    plt.ylabel("Mean elo_slope_per_game")
    plt.tight_layout()
    save_plot(fig, plots_dir / "elo_slope_mean_with_std_barplot.png", display_outputs=display_outputs)

    fig = plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=final_dataset_clustered,
        x="cluster",
        y="elo_slope_per_game"
    )
    plt.title(f"Distribution de elo_slope_per_game par cluster ({feature_set_name}, k={chosen_k})")
    plt.xlabel("Cluster")
    plt.ylabel("elo_slope_per_game")
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    save_plot(fig, plots_dir / "elo_slope_boxplot_with_outliers.png", display_outputs=display_outputs)

    fig = plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=final_dataset_clustered,
        x="cluster",
        y="elo_slope_per_game",
        showfliers=False
    )
    plt.title(f"Distribution de elo_slope_per_game par cluster sans outliers ({feature_set_name}, k={chosen_k})")
    plt.xlabel("Cluster")
    plt.ylabel("elo_slope_per_game")
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    save_plot(fig, plots_dir / "elo_slope_boxplot_without_outliers.png", display_outputs=display_outputs)

    return {"model_id": model_id, "model_dir": str(model_dir), "model_summary": model_summary}


In [26]:
render_reports = []
n_models = len(full_results)

for i, result in enumerate(full_results):
    display_outputs = DISPLAY_ALL_MODELS or (i < MAX_MODELS_TO_DISPLAY)
    report = render_and_save_model_outputs(
        result=result,
        output_root=OUTPUT_DIR,
        display_outputs=display_outputs,
    )
    render_reports.append(report)

print(f"Analyses détaillées générées pour {n_models} modèles.")
print(f"Sortie enregistrée dans : {OUTPUT_DIR.resolve()}")

Analyses détaillées générées pour 23 modèles.
Sortie enregistrée dans : C:\Projets_ETS\MTI830\clustering_chess_player_lichess-1\analysis_outputs\full_results_deep_dive


In [6]:
all_model_summaries = pd.DataFrame(
    [build_model_summary_from_result(res) for res in full_results]
)

all_model_summaries = all_model_summaries.sort_values(
    by=[
        "elo_slope_per_game_std_between_clusters_weighted",
        "silhouette",
        "elo_slope_per_game_std_within_clusters_weighted",
    ],
    ascending=[False, False, True]
).reset_index(drop=True)

display(all_model_summaries)
all_model_summaries.to_csv(OUTPUT_DIR / "all_model_summaries.csv", index=False)

selected_models_df = all_model_summaries.copy()
display(selected_models_df)
selected_models_df.to_csv(OUTPUT_DIR / "selected_models_comparison.csv", index=False)

,feature_set_name,k,best_seed,n_features,features,silhouette,inertia,min_cluster_size,max_cluster_size,imbalance_ratio,elo_slope_per_game_range,elo_slope_per_game_std_between_clusters,elo_slope_per_game_std_between_clusters_weighted,elo_slope_per_game_std_within_clusters_weighted
0,COMBO_5_mean_ply_count__delay_ratio_when_loses...,5,14,5,"[mean_ply_count, delay_ratio_when_losestreak, ...",0.183664,421.953432,51,78,0.653846,0.346945,0.119862,0.112294,0.356816
1,COMBO_5_mean_ply_count__draw_ratio__cv_games_p...,5,12,5,"[mean_ply_count, draw_ratio, cv_games_per_day,...",0.170983,459.845480,28,82,0.341463,0.316362,0.116517,0.110136,0.357811
2,COMBO_4_mean_ply_count__delay_ratio_when_winst...,5,10,4,"[mean_ply_count, delay_ratio_when_winstreak, e...",0.191400,343.472167,27,70,0.385714,0.283862,0.106374,0.109109,0.356787
3,COMBO_3_mean_ply_count__entropy_sessions_inter...,5,14,3,"[mean_ply_count, entropy_sessions_interval, me...",0.246181,188.878590,28,86,0.325581,0.276799,0.097656,0.108338,0.355340
4,COMBO_5_draw_ratio__cv_games_per_week__cv_game...,5,13,5,"[draw_ratio, cv_games_per_week, cv_games_inter...",0.182652,461.951006,22,84,0.261905,0.296023,0.114769,0.107447,0.355763
5,COMBO_4_mean_ply_count__cv_games_per_day__cv_g...,5,18,4,"[mean_ply_count, cv_games_per_day, cv_games_pe...",0.244178,293.333257,32,100,0.320000,0.352305,0.121177,0.105965,0.360211
6,COMBO_4_mean_ply_count__draw_ratio__cv_games_p...,5,19,4,"[mean_ply_count, draw_ratio, cv_games_per_day,...",0.202415,301.201426,43,94,0.457447,0.363422,0.119664,0.105423,0.358301
7,COMBO_4_mean_ply_count__cv_games_per_day__cv_g...,5,11,4,"[mean_ply_count, cv_games_per_day, cv_games_in...",0.196941,343.431544,41,91,0.450549,0.357611,0.117400,0.104904,0.360880
8,COMBO_5_mean_ply_count__draw_ratio__delay_rati...,4,8,5,"[mean_ply_count, draw_ratio, delay_ratio_when_...",0.170853,500.740326,70,76,0.921053,0.292698,0.103813,0.104835,0.358930
9,COMBO_3_mean_ply_count__cv_games_per_day__entr...,5,11,3,"[mean_ply_count, cv_games_per_day, entropy_ses...",0.240694,183.166571,43,80,0.537500,0.345227,0.112799,0.104757,0.357118


,feature_set_name,k,best_seed,n_features,features,silhouette,inertia,min_cluster_size,max_cluster_size,imbalance_ratio,elo_slope_per_game_range,elo_slope_per_game_std_between_clusters,elo_slope_per_game_std_between_clusters_weighted,elo_slope_per_game_std_within_clusters_weighted
0,COMBO_5_mean_ply_count__delay_ratio_when_loses...,5,14,5,"[mean_ply_count, delay_ratio_when_losestreak, ...",0.183664,421.953432,51,78,0.653846,0.346945,0.119862,0.112294,0.356816
1,COMBO_5_mean_ply_count__draw_ratio__cv_games_p...,5,12,5,"[mean_ply_count, draw_ratio, cv_games_per_day,...",0.170983,459.845480,28,82,0.341463,0.316362,0.116517,0.110136,0.357811
2,COMBO_4_mean_ply_count__delay_ratio_when_winst...,5,10,4,"[mean_ply_count, delay_ratio_when_winstreak, e...",0.191400,343.472167,27,70,0.385714,0.283862,0.106374,0.109109,0.356787
3,COMBO_3_mean_ply_count__entropy_sessions_inter...,5,14,3,"[mean_ply_count, entropy_sessions_interval, me...",0.246181,188.878590,28,86,0.325581,0.276799,0.097656,0.108338,0.355340
4,COMBO_5_draw_ratio__cv_games_per_week__cv_game...,5,13,5,"[draw_ratio, cv_games_per_week, cv_games_inter...",0.182652,461.951006,22,84,0.261905,0.296023,0.114769,0.107447,0.355763
5,COMBO_4_mean_ply_count__cv_games_per_day__cv_g...,5,18,4,"[mean_ply_count, cv_games_per_day, cv_games_pe...",0.244178,293.333257,32,100,0.320000,0.352305,0.121177,0.105965,0.360211
6,COMBO_4_mean_ply_count__draw_ratio__cv_games_p...,5,19,4,"[mean_ply_count, draw_ratio, cv_games_per_day,...",0.202415,301.201426,43,94,0.457447,0.363422,0.119664,0.105423,0.358301
7,COMBO_4_mean_ply_count__cv_games_per_day__cv_g...,5,11,4,"[mean_ply_count, cv_games_per_day, cv_games_in...",0.196941,343.431544,41,91,0.450549,0.357611,0.117400,0.104904,0.360880
8,COMBO_5_mean_ply_count__draw_ratio__delay_rati...,4,8,5,"[mean_ply_count, draw_ratio, delay_ratio_when_...",0.170853,500.740326,70,76,0.921053,0.292698,0.103813,0.104835,0.358930
9,COMBO_3_mean_ply_count__cv_games_per_day__entr...,5,11,3,"[mean_ply_count, cv_games_per_day, entropy_ses...",0.240694,183.166571,43,80,0.537500,0.345227,0.112799,0.104757,0.357118


In [14]:
full_results_dict = {
    (res["feature_set_name"], int(res["k"])): res
    for res in full_results
}

# print("Exemple de clés disponibles :")
# for key in list(full_results_dict.keys())[:24]:
#     print(key)

id_entry = 0

example_feature_set_name = all_model_summaries.loc[id_entry, "feature_set_name"]
example_k = int(all_model_summaries.loc[id_entry, "k"])

selected_result = full_results_dict[(example_feature_set_name, example_k)]

display(Markdown(f"## Inspection ciblée : {example_feature_set_name} (k={example_k})"))
display(pd.DataFrame([build_model_summary_from_result(selected_result)]))
display(selected_result["cluster_sizes"])
display(selected_result["progression_summary_mean"])
display(selected_result["progression_summary_median"])
display(selected_result["clustering_summary_mean"])
display(selected_result["standardized_profiles"])

## Inspection ciblée : COMBO_5_mean_ply_count__delay_ratio_when_losestreak__cv_games_per_day__cv_games_per_week__entropy_sessions_interval (k=5)

,feature_set_name,k,best_seed,n_features,features,silhouette,inertia,min_cluster_size,max_cluster_size,imbalance_ratio,elo_slope_per_game_range,elo_slope_per_game_std_between_clusters,elo_slope_per_game_std_between_clusters_weighted,elo_slope_per_game_std_within_clusters_weighted
0,COMBO_5_mean_ply_count__delay_ratio_when_loses...,5,14,5,"[mean_ply_count, delay_ratio_when_losestreak, ...",0.183664,421.953432,51,78,0.653846,0.346945,0.119862,0.112294,0.356816


cluster
0    78
1    51
2    51
3    53
4    65
Name: size, dtype: int64

,elo_gain,elo_gain_per_game,elo_slope_per_game,elo_slope_per_day
cluster,,,,
0,14.692308,0.043936,0.118290,0.540092
1,67.411765,0.134342,0.271787,0.960805
2,2.156863,0.021602,0.219899,0.491600
3,-50.150943,-0.102123,-0.075158,-0.183436
4,17.676923,0.019140,0.090737,0.460831


,elo_gain,elo_gain_per_game,elo_slope_per_game,elo_slope_per_day
cluster,,,,
0,-1.5,-0.003162,0.081502,0.354140
1,49.0,0.111531,0.226084,0.798907
2,13.0,0.027500,0.127269,0.413173
3,-41.0,-0.073913,-0.054303,-0.175937
4,-1.0,-0.004386,0.108996,0.368007


,mean_ply_count,delay_ratio_when_losestreak,cv_games_per_day,cv_games_per_week,entropy_sessions_interval
cluster,,,,,
0,62.191337,1.057312,0.630195,0.540923,0.936141
1,63.692487,0.610500,0.863708,0.857245,1.191531
2,54.823244,1.348775,0.965437,1.064288,1.274943
3,51.383855,0.884917,0.760708,0.676281,1.077231
4,60.182185,1.773589,0.796046,0.757879,1.107511


,mean_ply_count,delay_ratio_when_losestreak,cv_games_per_day,cv_games_per_week,entropy_sessions_interval
cluster,,,,,
0,0.805636,-0.194611,-1.556694,-1.353199,-1.591827
1,1.129284,-1.313620,0.544222,0.442295,0.650140
2,-0.782925,0.535339,1.459479,1.617505,1.382379
3,-1.524457,-0.626362,-0.382470,-0.584884,-0.353252
4,0.372463,1.599254,-0.064536,-0.121718,-0.087439
